In [75]:
import pandas as pd
import ast

In [ ]:
detail = pd.read_csv("crawl_data/foody_detail_data.csv")
branch = pd.read_csv("crawl_data/foody_branch_data.csv")
search = pd.read_csv("crawl_data/foody_search_data.csv")
hours = pd.read_csv("crawl_data/foody_store_opening_hours.csv")

In [77]:
print(detail['AvgPointList'].iloc[0])

[{'Label': 'Vị trí', 'Point': 0.0}, {'Label': 'Giá cả', 'Point': 0.0}, {'Label': 'Chất lượng', 'Point': 0.0}, {'Label': 'Phục vụ', 'Point': 0.0}, {'Label': 'Không gian', 'Point': 0.0}]


In [78]:
detail_copy = detail.copy()

def flatten_points(val):
    if pd.isna(val): return {}
    if isinstance(val, str):
        try:
            data_list = ast.literal_eval(val)
        except:
            return {}
    else:
        data_list = val
    
    return {item['Label']: item['Point'] for item in data_list}

temp = detail_copy['AvgPointList'].apply(flatten_points)
points_df = pd.DataFrame(temp.tolist(), index=detail_copy.index)
detail_copy = pd.concat([detail_copy.drop(columns=['AvgPointList']), points_df], axis=1)

In [79]:
print(detail_copy.iloc[0])

PictureModel         {'imageUrl': 'https://down-vn.img.susercontent...
HasValidContract                                                 False
TableNowApiEnable                                                 True
RestaurantID                                              1000025953.0
Name                 Con Cú Có Cacao - Chuyên Trà Sữa Từ Sữa Tươi -...
Address              114 Chung Cư Đoàn Văn Bơ (Hẻm 83 Hoàng Diệu), ...
District                                                        Quận 4
Area                                                               NaN
AreaId                                                             NaN
City                                                           TP. HCM
CityId                                                             NaN
PriceMin                                                           0.0
PriceMax                                                           0.0
IsActive                                                         False
MetaTi

In [80]:
print(detail['Cuisines'].iloc[0])

[{'UrlRewriteName': 'viet-nam', 'Name': 'Món Việt', 'NameEn': 'Vietnam', 'DisplayOrder': 0, 'Flag': 0, 'ParentID': 0, 'Icon': 'foody-01-vietnam-635950402447733778.png', 'Id': 1}]


In [ ]:
def extract_names(cuisine_str):
    if pd.isna(cuisine_str) or cuisine_str == "" or cuisine_str == "[]":
        return None
        
    try:
        data = ast.literal_eval(cuisine_str)
        
        if isinstance(data, list):
            names = [item.get('Name') for item in data if isinstance(item, dict) and item.get('Name')]
            
            return " || ".join(names) if names else None
            
        elif isinstance(data, dict):
            return data.get('Name')
            
    except (ValueError, SyntaxError):
        return None
    return None

detail_copy['Cuisines'] = detail_copy['Cuisines'].apply(extract_names)

In [82]:
print(detail_copy['Cuisines'].iloc[0])

Món Việt


In [ ]:
def format_target_audience(val_str):
    try:
        data = ast.literal_eval(val_str)
        
        names = [item['Name'] for item in data]
        
        return " || ".join(names)
    except (ValueError, SyntaxError, TypeError):
        return ""

detail_copy['LstTargetAudience'] = detail_copy['LstTargetAudience'].apply(format_target_audience)

In [84]:
print(detail_copy['LstTargetAudience'].iloc[0])

Sinh viên || Cặp đôi || Nhóm hội


In [ ]:
def extract_category_names(category_str):
    if pd.isna(category_str) or category_str == "" or category_str == "[]":
        return None
        
    try:
        data = ast.literal_eval(category_str)
        
        if isinstance(data, list):
            names = [item.get('Name') for item in data if isinstance(item, dict) and item.get('Name')]
            return " || ".join(names) if names else None
            
        elif isinstance(data, dict):
            return data.get('Name')
            
    except (ValueError, SyntaxError):
        return None
    return None

detail_copy['LstCategory'] = detail_copy['LstCategory'].apply(extract_category_names)

In [86]:
print(detail_copy['LstCategory'].iloc[0])

Ăn vặt/vỉa hè || Café/Dessert


In [ ]:
def transform_ratings_to_columns(rating_str):
    default = {'Excellent': 0, 'Good': 0, 'Average': 0, 'Bad': 0}
    
    if pd.isna(rating_str) or rating_str == "" or rating_str == "{}":
        return pd.Series(default)
        
    try:
        data = ast.literal_eval(rating_str) if isinstance(rating_str, str) else rating_str
        
        if isinstance(data, dict):
            return pd.Series({
                'Excellent': int(data.get('excelent', 0)), 
                'Good': int(data.get('good', 0)),
                'Average': int(data.get('average', 0)),
                'Bad': int(data.get('bad', 0))
            })
            
    except (ValueError, SyntaxError, TypeError):
        return pd.Series(default)
        
    return pd.Series(default)

ratings_expanded = detail_copy['Ratings'].apply(transform_ratings_to_columns)
detail_copy = pd.concat([detail_copy.drop(columns=['Ratings']), ratings_expanded], axis=1)

In [88]:
def extract_url(gia_tri):
    if pd.isna(gia_tri): 
        return gia_tri
    try:
        dict_data = ast.literal_eval(str(gia_tri))
        return dict_data.get('imageUrl')
    except (ValueError, SyntaxError):
        return None
    
detail_copy['PictureModel'] = detail_copy['PictureModel'].apply(extract_url)

print(detail_copy['PictureModel'])

0         https://down-vn.img.susercontent.com/vn-111345...
1         https://down-vn.img.susercontent.com/vn-111342...
2         https://down-vn.img.susercontent.com/vn-111342...
3         https://down-vn.img.susercontent.com/vn-111342...
4         https://down-vn.img.susercontent.com/vn-111342...
                                ...                        
107360    https://down-vn.img.susercontent.com/vn-111342...
107361    https://down-vn.img.susercontent.com/vn-111342...
107362    https://down-vn.img.susercontent.com/vn-111342...
107363    https://down-vn.img.susercontent.com/vn-111342...
107364    https://down-vn.img.susercontent.com/vn-111342...
Name: PictureModel, Length: 107365, dtype: object


In [89]:
print(detail_copy.columns)

Index(['PictureModel', 'HasValidContract', 'TableNowApiEnable', 'RestaurantID',
       'Name', 'Address', 'District', 'Area', 'AreaId', 'City', 'CityId',
       'PriceMin', 'PriceMax', 'IsActive', 'MetaTitle', 'MetaKeywords',
       'MetaDescription', 'Cuisines', 'LstTargetAudience', 'LstCategory',
       'Services', 'AccessGuide', 'DeliveryId', 'RestaurantUrl',
       'ResUrlRewrite', 'Properties', 'MiscInfo', 'Vị trí', 'Giá cả',
       'Chất lượng', 'Phục vụ', 'Không gian', 'Excellent', 'Good', 'Average',
       'Bad'],
      dtype='object')


In [90]:
print(detail_copy.iloc[0])

PictureModel         https://down-vn.img.susercontent.com/vn-111345...
HasValidContract                                                 False
TableNowApiEnable                                                 True
RestaurantID                                              1000025953.0
Name                 Con Cú Có Cacao - Chuyên Trà Sữa Từ Sữa Tươi -...
Address              114 Chung Cư Đoàn Văn Bơ (Hẻm 83 Hoàng Diệu), ...
District                                                        Quận 4
Area                                                               NaN
AreaId                                                             NaN
City                                                           TP. HCM
CityId                                                             NaN
PriceMin                                                           0.0
PriceMax                                                           0.0
IsActive                                                         False
MetaTi

In [91]:
drop_columns = ['HasValidContract', 'TableNowApiEnable', 'AreaId', 'City', 'CityId', 'MetaTitle', 'MetaDescription', 'ResUrlRewrite', 'Properties', 'MiscInfo', 'AccessGuide']
detail_copy.drop(columns=drop_columns, inplace=True)

In [92]:
print(detail_copy.shape)

(107365, 25)


In [93]:
print(detail_copy.iloc[0])

PictureModel         https://down-vn.img.susercontent.com/vn-111345...
RestaurantID                                              1000025953.0
Name                 Con Cú Có Cacao - Chuyên Trà Sữa Từ Sữa Tươi -...
Address              114 Chung Cư Đoàn Văn Bơ (Hẻm 83 Hoàng Diệu), ...
District                                                        Quận 4
Area                                                               NaN
PriceMin                                                           0.0
PriceMax                                                           0.0
IsActive                                                         False
MetaKeywords                                        Trà sữa, Trà Chanh
Cuisines                                                      Món Việt
LstTargetAudience                     Sinh viên || Cặp đôi || Nhóm hội
LstCategory                              Ăn vặt/vỉa hè || Café/Dessert
Services                                                            []
Delive

In [94]:
print(detail_copy.shape)

(107365, 25)


In [95]:
print(search.columns)

Index(['Id', 'Name', 'Address', 'District', 'City', 'Phone',
       'AvgRatingOriginal', 'Cuisines', 'AlbumUrl', 'HasDelivery',
       'HasBooking', 'BookingUrl', 'HasVideo', 'Services', 'Categories',
       'DetailUrl', 'BranchUrl', 'TotalReviews', 'TotalView', 'TotalFavourite',
       'TotalCheckins', 'Status', 'review_url', 'PicturePath',
       'PicturePathLarge', 'MobilePicturePath', 'SubItems', 'MainCategoryId',
       'totalResult'],
      dtype='object')


In [ ]:
df_combine = pd.merge(
    detail_copy, 
    search[['Id', 'Services']], 
    left_on='RestaurantID', 
    right_on='Id', 
    how='left', 
    suffixes=('_A', '_B')
)

def fill_services(row):
    val_a = row['Services_A']
    val_b = row['Services_B']
    
    is_empty_a = pd.isna(val_a) or val_a == "" or val_a == "[]" or val_a == []
    
    if is_empty_a:
        return val_b 
    else:
        return val_a 
    
df_combine['Services'] = df_combine.apply(fill_services, axis=1)
df_combine.drop(columns=['Services_A', 'Services_B', 'Id'], inplace=True)

In [97]:
print(df_combine.shape)

(107365, 25)


In [ ]:
detail_combine = pd.merge(
    df_combine, 
    search[['Id', 'TotalReviews', 'TotalView', 'TotalFavourite', 'TotalCheckins']], 
    left_on='RestaurantID', 
    right_on='Id',
    how='left'
)

detail_combine = detail_combine.drop(columns=['Id'])
print(detail_combine[['TotalReviews', 'TotalView', 'TotalFavourite', 'TotalCheckins']].head())

   TotalReviews  TotalView  TotalFavourite  TotalCheckins
0           NaN       12.0             0.0            0.0
1           NaN      607.0             1.0            1.0
2           NaN      321.0             0.0            0.0
3           NaN      201.0             2.0            0.0
4           NaN      407.0             5.0            0.0


In [99]:
print(detail_combine['Services'].head())

0    [{'Name': 'Giao tận nơi', 'Id': 1}]
1                                     []
2                                     []
3    [{'Name': 'Giao tận nơi', 'Id': 1}]
4                                     []
Name: Services, dtype: object


In [100]:
print(detail_combine['Services'].iloc[0]) 
print(type(detail_combine['Services'].iloc[0]))

[{'Name': 'Giao tận nơi', 'Id': 1}]
<class 'str'>


In [ ]:
unique_services = detail_combine['Services'].dropna().unique()

unique_services = [s for s in unique_services if str(s).strip() != "" and s != 'None']

print(f"Số lượng giá trị unique Name: {len(unique_services)}")
print("Danh sách các Services:")
print(unique_services)

Số lượng giá trị unique Name: 4
Danh sách các Services:
["[{'Name': 'Giao tận nơi', 'Id': 1}]", '[]', "[{'Name': 'Đặt bàn ', 'Id': 2}]", "[{'Name': 'Đặt bàn ', 'Id': 2}, {'Name': 'Giao tận nơi', 'Id': 1}]"]


In [ ]:
def one_hot_services(row):
    try:
        services_list = ast.literal_eval(row)
        names = [s['Name'].strip() for s in services_list]
        
        return pd.Series({
            'Giao tận nơi': 1 if 'Giao tận nơi' in names else 0,
            'Đặt bàn': 1 if 'Đặt bàn' in names else 0
        })
    except:
        return pd.Series({'Giao tận nơi': 0, 'Đặt bàn': 0})

services_one_hot = detail_combine['Services'].apply(one_hot_services)
detail_combine = pd.concat([detail_combine, services_one_hot], axis=1)

In [103]:
print(detail_combine.iloc[0])

PictureModel         https://down-vn.img.susercontent.com/vn-111345...
RestaurantID                                              1000025953.0
Name                 Con Cú Có Cacao - Chuyên Trà Sữa Từ Sữa Tươi -...
Address              114 Chung Cư Đoàn Văn Bơ (Hẻm 83 Hoàng Diệu), ...
District                                                        Quận 4
Area                                                               NaN
PriceMin                                                           0.0
PriceMax                                                           0.0
IsActive                                                         False
MetaKeywords                                        Trà sữa, Trà Chanh
Cuisines                                                      Món Việt
LstTargetAudience                     Sinh viên || Cặp đôi || Nhóm hội
LstCategory                              Ăn vặt/vỉa hè || Café/Dessert
DeliveryId                                                    185081.0
Restau

In [104]:
print(detail_combine.columns)

Index(['PictureModel', 'RestaurantID', 'Name', 'Address', 'District', 'Area',
       'PriceMin', 'PriceMax', 'IsActive', 'MetaKeywords', 'Cuisines',
       'LstTargetAudience', 'LstCategory', 'DeliveryId', 'RestaurantUrl',
       'Vị trí', 'Giá cả', 'Chất lượng', 'Phục vụ', 'Không gian', 'Excellent',
       'Good', 'Average', 'Bad', 'Services', 'TotalReviews', 'TotalView',
       'TotalFavourite', 'TotalCheckins', 'Giao tận nơi', 'Đặt bàn'],
      dtype='object')


In [105]:
print(detail_combine.iloc[0])

PictureModel         https://down-vn.img.susercontent.com/vn-111345...
RestaurantID                                              1000025953.0
Name                 Con Cú Có Cacao - Chuyên Trà Sữa Từ Sữa Tươi -...
Address              114 Chung Cư Đoàn Văn Bơ (Hẻm 83 Hoàng Diệu), ...
District                                                        Quận 4
Area                                                               NaN
PriceMin                                                           0.0
PriceMax                                                           0.0
IsActive                                                         False
MetaKeywords                                        Trà sữa, Trà Chanh
Cuisines                                                      Món Việt
LstTargetAudience                     Sinh viên || Cặp đôi || Nhóm hội
LstCategory                              Ăn vặt/vỉa hè || Café/Dessert
DeliveryId                                                    185081.0
Restau

In [ ]:
detail_combine_result = pd.merge(
    detail_combine, 
    hours[['Id', 'Chủ nhật', 'Thứ hai', 'Thứ ba', 'Thứ tư', 'Thứ năm', 'Thứ sáu', 'Thứ bảy']], 
    left_on='RestaurantID', 
    right_on='Id',
    how='left'
)

detail_combine_result = detail_combine_result.drop(columns=['Id'])
print(detail_combine_result[['Chủ nhật', 'Thứ hai', 'Thứ ba', 'Thứ tư', 'Thứ năm', 'Thứ sáu', 'Thứ bảy']].head())

                                            Chủ nhật  \
0  {'time': [{'open': '16:30', 'close': '23:59', ...   
1  {'time': [{'open': '17:00', 'close': '00:00', ...   
2  {'time': [{'open': '07:00', 'close': '22:00', ...   
3  {'time': [{'open': '06:00', 'close': '22:00', ...   
4  {'time': [{'open': '09:00', 'close': '21:00', ...   

                                             Thứ hai  \
0  {'time': [{'open': '16:30', 'close': '23:59', ...   
1  {'time': [{'open': '17:00', 'close': '00:00', ...   
2  {'time': [{'open': '07:00', 'close': '22:00', ...   
3  {'time': [{'open': '06:00', 'close': '22:00', ...   
4  {'time': [{'open': '09:00', 'close': '21:00', ...   

                                              Thứ ba  \
0  {'time': [{'open': '16:30', 'close': '23:59', ...   
1  {'time': [{'open': '17:00', 'close': '00:00', ...   
2  {'time': [{'open': '07:00', 'close': '22:00', ...   
3  {'time': [{'open': '06:00', 'close': '22:00', ...   
4  {'time': [{'open': '09:00', 'close': '21:00

In [107]:
print(detail_combine_result.columns)

Index(['PictureModel', 'RestaurantID', 'Name', 'Address', 'District', 'Area',
       'PriceMin', 'PriceMax', 'IsActive', 'MetaKeywords', 'Cuisines',
       'LstTargetAudience', 'LstCategory', 'DeliveryId', 'RestaurantUrl',
       'Vị trí', 'Giá cả', 'Chất lượng', 'Phục vụ', 'Không gian', 'Excellent',
       'Good', 'Average', 'Bad', 'Services', 'TotalReviews', 'TotalView',
       'TotalFavourite', 'TotalCheckins', 'Giao tận nơi', 'Đặt bàn',
       'Chủ nhật', 'Thứ hai', 'Thứ ba', 'Thứ tư', 'Thứ năm', 'Thứ sáu',
       'Thứ bảy'],
      dtype='object')


In [108]:
print(detail_combine_result.iloc[0])

PictureModel         https://down-vn.img.susercontent.com/vn-111345...
RestaurantID                                              1000025953.0
Name                 Con Cú Có Cacao - Chuyên Trà Sữa Từ Sữa Tươi -...
Address              114 Chung Cư Đoàn Văn Bơ (Hẻm 83 Hoàng Diệu), ...
District                                                        Quận 4
Area                                                               NaN
PriceMin                                                           0.0
PriceMax                                                           0.0
IsActive                                                         False
MetaKeywords                                        Trà sữa, Trà Chanh
Cuisines                                                      Món Việt
LstTargetAudience                     Sinh viên || Cặp đôi || Nhóm hội
LstCategory                              Ăn vặt/vỉa hè || Café/Dessert
DeliveryId                                                    185081.0
Restau

In [ ]:
def transform_opening_hours(val_str):
    if pd.isna(val_str) or val_str == "" or val_str == "{}":
        return (("00:00", "00:00"), True) 
        
    try:
        data = ast.literal_eval(val_str) if isinstance(val_str, str) else val_str
        time_info = data.get('time', [{}])[0]
        open_time = time_info.get('open', '00:00')
        close_time = time_info.get('close', '00:00')
        is_day_off = bool(data.get('is_day_off', False))
        
        return ((open_time, close_time), is_day_off)
        
    except (ValueError, SyntaxError, IndexError, AttributeError):
        return (("00:00", "00:00"), True)

days_cols = ['Chủ nhật', 'Thứ hai', 'Thứ ba', 'Thứ tư', 'Thứ năm', 'Thứ sáu', 'Thứ bảy']

for col in days_cols:
    detail_combine_result[col] = detail_combine_result[col].apply(transform_opening_hours)

In [110]:
print(detail_combine_result['Chủ nhật'].head())

0    ((16:30, 23:59), False)
1    ((17:00, 00:00), False)
2    ((07:00, 22:00), False)
3    ((06:00, 22:00), False)
4    ((09:00, 21:00), False)
Name: Chủ nhật, dtype: object


In [ ]:
def extract_day_off(row):
    off_days = []
    for col in days_cols:
        if row[col][1] == True:
            off_days.append(col)
    return " || ".join(off_days)

detail_combine_result['Ngày nghỉ'] = detail_combine_result.apply(extract_day_off, axis=1)

for col in days_cols:
    detail_combine_result[col] = detail_combine_result[col].apply(lambda x: x[0])

In [112]:
print(detail_combine_result[detail_combine_result['RestaurantID'] == 1000075272].iloc[0])

PictureModel         https://down-vn.img.susercontent.com/vn-111345...
RestaurantID                                              1000075272.0
Name                        Gau Bon - Fresh Juice - Đông Hưng Thuận 42
Address                    67/7 Đông Hưng Thuận 42, P. Tân Hưng Thuận
District                                                       Quận 12
Area                                                               NaN
PriceMin                                                           0.0
PriceMax                                                           0.0
IsActive                                                         False
MetaKeywords                                                       NaN
Cuisines                                                      Món Việt
LstTargetAudience                                                     
LstCategory                                                Shop Online
DeliveryId                                                    341303.0
Restau

In [113]:
print(detail_combine_result.iloc[0])

PictureModel         https://down-vn.img.susercontent.com/vn-111345...
RestaurantID                                              1000025953.0
Name                 Con Cú Có Cacao - Chuyên Trà Sữa Từ Sữa Tươi -...
Address              114 Chung Cư Đoàn Văn Bơ (Hẻm 83 Hoàng Diệu), ...
District                                                        Quận 4
Area                                                               NaN
PriceMin                                                           0.0
PriceMax                                                           0.0
IsActive                                                         False
MetaKeywords                                        Trà sữa, Trà Chanh
Cuisines                                                      Món Việt
LstTargetAudience                     Sinh viên || Cặp đôi || Nhóm hội
LstCategory                              Ăn vặt/vỉa hè || Café/Dessert
DeliveryId                                                    185081.0
Restau

In [114]:
print(detail.shape)
print(detail_combine_result.shape)

(107365, 29)
(107365, 39)


In [115]:
detail_combine_result.drop(columns=['Services', 'TotalReviews', 'IsActive'], inplace=True)

In [116]:
print(detail_combine_result.iloc[0])

PictureModel         https://down-vn.img.susercontent.com/vn-111345...
RestaurantID                                              1000025953.0
Name                 Con Cú Có Cacao - Chuyên Trà Sữa Từ Sữa Tươi -...
Address              114 Chung Cư Đoàn Văn Bơ (Hẻm 83 Hoàng Diệu), ...
District                                                        Quận 4
Area                                                               NaN
PriceMin                                                           0.0
PriceMax                                                           0.0
MetaKeywords                                        Trà sữa, Trà Chanh
Cuisines                                                      Món Việt
LstTargetAudience                     Sinh viên || Cặp đôi || Nhóm hội
LstCategory                              Ăn vặt/vỉa hè || Café/Dessert
DeliveryId                                                    185081.0
RestaurantUrl        ho-chi-minh/con-cu-co-cacao-chuyen-tra-sua-tu-...
Vị trí

In [ ]:
detail_combine_result.to_csv("../dataset/foody_combined_data_final.csv", encoding='utf-8-sig', index=False)